In [1]:
import os
import numpy as np
from datasets import load_dataset
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tenacity import retry, wait_random_exponential
from tqdm.asyncio import tqdm_asyncio
import asyncio
import json
from pathlib import Path

In [2]:
load_dotenv("secret.env")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
DATASET = "ddz5431/refact"
N = 100
SEED = 42
CACHE = Path("cache")

# Load Dataset

In [3]:
ds = load_dataset(DATASET, split="train").to_pandas()
samples = ds.sample(n=N, random_state=SEED).reset_index(drop=True)

# Randomly assign which answer is A and which is B to avoid position bias
rng = np.random.default_rng(SEED)
samples["correct_is_a"] = rng.integers(0, 2, size=N).astype(bool)
samples["answer_a"] = samples.apply(
    lambda r: r.correct_answer if r.correct_is_a else r.confabulated_answer, axis=1
)
samples["answer_b"] = samples.apply(
    lambda r: r.confabulated_answer if r.correct_is_a else r.correct_answer, axis=1
)
samples[["sample_id", "question", "error_type", "correct_is_a"]]

,sample_id,question,error_type,correct_is_a
0,e734464a0b7f1607b098c797432f6c59f1b823cb025d1a...,How are gas giants spheres? Wouldn’t the gas j...,swap,False
1,d62cb113c635c6e2f886e1fbe2b6879061497b60e1b07c...,A question about particle collisions In a coll...,neg,True
2,574579ab73152181b935562b6363248e89fb258baf42f8...,Do fish get “out of breath”?,neg,True
3,f3f417a43f1aa817e2589aa3847799f6655d1ab8c6ac26...,Some scientist hypothesis that Solar Wind stri...,neg,False
4,b76a82e8dd70c0cdad4e882abe06c5d9786942284438a5...,Does the James Webb telescope have an increase...,neg,False
...,...,...,...,...
95,c0a0b9ba1f12e95eb1481f9104166383b214a1b267c9c1...,What is the correlation between brain size and...,neg,False
96,1485544336eff471f5e8cbac45d3ad3c1fcfcbdf9d8b7f...,If we can discharge excess voltage into the Ea...,swap,False
97,29dd4e42dafb3fdd52dbc35016352e4ce585ac231e196e...,How does a zygote transcribe its DNA initially...,swap,True
98,21a06560adc476548570e011842ff466c3c9d804d0ecfa...,"If all electrons are identical, then how can t...",swap,True


# System Prompts

In [4]:
SYSTEM_PROMPT_BASE = """\
Your task is to return the factually correct answer out of the two given answers (A and B) \
to a question. Read both entire answers and the question carefully, then analyze the factual \
accuracy of both answers within the context. \
After your analysis, provide a final verdict of either answer A or answer B is factually correct. \
Output exactly one of:
   FINAL VERDICT: A
   FINAL VERDICT: B
at the very end to mark the correct answer.
"""

SYSTEM_PROMPT_COMP = """\
You are a scientific fact-checker tasked with identifying which of two answers to a science question is factually correct, and which contains a confabulation.

A confabulation is a subtle, plausible-sounding substitution — either a logical negation that flips the direction of a true claim ("lose" → "gain", "blocks" → "encourages"), or a domain-specific term swapped for a wrong but plausible-sounding alternative ("Sodium/Potassium ATPase" → "Calcium/Magnesium Pump").

The two answers you receive are nearly identical. They differ in at most a few words or one phrase. Your job is to find that difference and determine which version states something true.

---

CRITICAL RULES

1. EXACTLY ONE ANSWER IS CORRECT
   Never conclude that both are wrong or both are correct. Commit to a verdict.

2. DIFF FIRST, EVALUATE SECOND
   Do not evaluate each answer independently from scratch. Read both answers together, locate every point of disagreement, and only then assess which version of each differing claim is correct.

3. PLAUSIBLE SUBSTITUTIONS ARE INTENTIONAL
   In entity swaps, the substituted term is chosen to sound domain-appropriate. Verify it against your knowledge; do not accept a technical-sounding term just because it fits grammatically.

---

PROCEDURE

1. DIFF
   Quote every passage where Answer A and Answer B disagree, in the form:
     A: "..."
     B: "..."

2. EVALUATE
   For each differing passage, state which version is factually correct and briefly explain why. If one version is internally inconsistent with surrounding text, note that as well.

3. VERDICT
   Output exactly one of:
     FINAL VERDICT: A
     FINAL VERDICT: B

---

EXAMPLE 1

Question: Why is it that some people can hear the high frequency sound of a turned-on television and others can't? Is it genetics? Environment? Do people raised in a more rural environment keep their sensitivities to high frequency sounds?

Answer A: It's age and genetics. Some people can simply hear better, it's standardly distributed. As we get older we also lose the ability to hear very high and low sounds. Normal hearing is around 20-20000 Hz. If you hear the whining sounds others can't you can be grateful you have more acute hearing than they do. Alternately if you're hearing them often and nobody else does or can't figure out where it's coming from you may have tinnitus.

Answer B: It's age and genetics. Some people can simply hear better, it's standardly distributed. As we get older we gain the ability to hear very high and low sounds. Normal hearing is around 20-20000 Hz. If you hear the whining sounds others can't you can be grateful you have more acute hearing than they do. Alternately if you're hearing them often and nobody else does or can't figure out where it's coming from you may have tinnitus.

DIFF
  A: "As we get older we also lose the ability to hear very high and low sounds."
  B: "As we get older we gain the ability to hear very high and low sounds."

EVALUATE
  The claim concerns how hearing changes with age. Age-related hearing loss (presbycusis) is well-established: as people age, sensitivity to high-frequency sounds declines — it is not gained. "Lose" (A) is correct; "gain" (B) inverts the direction of a true claim and is factually wrong.

FINAL VERDICT: A

---

EXAMPLE 2

Question: If you had a perfectly reflective mirror on a frictionless track and you shined a flashlight on it, what would happen? Would the mirror move? Would the light reflect back?

Answer A: The light will get reflected back, and the mirror will start to move in the opposite direction. The reflected light will appear redder than the incident light due to blueshift.

Answer B: The light will get reflected back, and the mirror will start to move in the opposite direction. The reflected light will appear redder than the incident light due to redshift.

DIFF
  A: "...due to blueshift."
  B: "...due to redshift."

EVALUATE
  Both answers agree the reflected light appears redder. Redshift denotes a shift toward longer, redder wavelengths — consistent with reddening. Blueshift denotes a shift toward shorter, bluer wavelengths — the opposite, and contradicts the stated reddening. Answer A is internally inconsistent: it claims the light appears redder while attributing this to blueshift. The mirror moving away from the source causes a Doppler red-shift in the reflected light, confirming B.

FINAL VERDICT: B
\
"""

# Get GPT-4o answers (only if not cached yet)

In [5]:
# Semaphore caps in-flight requests to avoid bursting past the rate limit.
# Optimal size ≈ (RPM / 60) × avg_response_seconds — check your tier at:
# https://platform.openai.com/settings/organization/limits
MAX_CONCURRENT = 4

client = AsyncOpenAI(api_key=OPENAI_API_KEY)
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

USER_PROMPT = """\
Question: {question}
Answer A: {answer_a}
Answer B: {answer_b}
Final Verdict:\
"""

@retry(wait=wait_random_exponential(min=1, max=60))
async def call_gpt_4o(sample_id, correct_is_a, question, answer_a, answer_b, system_prompt):
    async with semaphore:
        judgement = await client.responses.create(
            model="gpt-4o",
            instructions=system_prompt,
            input=USER_PROMPT.format(question=question, answer_a=answer_a, answer_b=answer_b)
        )
        return {
            "sample_id": sample_id,
            "correct_is_a": correct_is_a,
            "judgement": judgement.output_text
        }

async def gather_results(samples, system_prompt):
    tasks = [
        call_gpt_4o(row.sample_id, row.correct_is_a, row.question, row.answer_a, row.answer_b, system_prompt)
        for _, row in samples.iterrows()
    ]
    return await tqdm_asyncio.gather(*tasks)

async def judge_samples_cached(cache_filename, system_prompt, samples):
    cache_file = CACHE / cache_filename
    if os.path.isfile(cache_file):
        print(f"Using cached results {cache_file}")
        with open(cache_file, "r") as f:
            return json.load(f)
    print("Gathering results from API")
    results = await gather_results(samples, system_prompt)
    with open(cache_file, "w") as f:
        json.dump(results, f)
    print(f"Written to {cache_file}")
    return results

results_base = await judge_samples_cached("comp_judgement_base.json", SYSTEM_PROMPT_BASE, samples)
results_comp = await judge_samples_cached("comp_judgement_comp.json", SYSTEM_PROMPT_COMP, samples)

Using cached results cache\comp_judgement_base.json
Using cached results cache\comp_judgement_comp.json


# Classify answers

In [6]:
def extract_verdict(judgement):
    text = judgement.lower()
    # find last occurrence of "answer a"/"answer b" or standalone "a"/"b" after "verdict"
    last_a = max(text.rfind("answer a"), text.rfind("verdict: a"), text.rfind("verdict is a"))
    last_b = max(text.rfind("answer b"), text.rfind("verdict: b"), text.rfind("verdict is b"))
    if last_a == last_b == -1:
        print(f"Cannot parse verdict from: {judgement!r}")
        return None
    return "A" if last_a > last_b else "B"

def classify(results):
    correct = total = 0
    for res in results:
        verdict = extract_verdict(res["judgement"])
        if verdict is None:
            continue
        predicted_correct_is_a = (verdict == "A")
        if predicted_correct_is_a == res["correct_is_a"]:
            correct += 1
        total += 1
    return correct, total

for label, results in [("Base prompt", results_base), ("Comp prompt", results_comp)]:
    correct, total = classify(results)
    print(f"{label}: {correct}/{total} correct ({correct/total:.0%})")

Base prompt: 94/100 correct (94%)
Comp prompt: 93/100 correct (93%)


# Example answers

In [7]:
def print_multi(title, text):
    SEP = "=============================================="
    print(SEP)
    print(title.upper())
    print(SEP)
    print(text)
    print(SEP)
    print()

def print_sample(sample_id, results, label=""):
    res = next(r for r in results if r["sample_id"] == sample_id)
    row = samples[samples["sample_id"] == sample_id].iloc[0]
    verdict = extract_verdict(res["judgement"])
    correct = "✓" if (verdict == "A") == row.correct_is_a else "✗"
    print_multi(f"[{label}] question", row.question)
    print_multi(f"answer A ({'CORRECT' if row.correct_is_a else 'CONFABULATED'})", row.answer_a)
    print_multi(f"answer B ({'CONFABULATED' if row.correct_is_a else 'CORRECT'})", row.answer_b)
    print_multi(f"judgement → {verdict} {correct}", res["judgement"])

sample_id = samples.iloc[0].sample_id
print_sample(sample_id, results_base, "Base")
print_sample(sample_id, results_comp, "Comp")

[BASE] QUESTION
How are gas giants spheres? Wouldn’t the gas just go everywhere in space? Especially Saturn, not only is it gas but it has a ring around it too. How is it not just a bunch of hot gas going in every direction?

ANSWER A (CONFABULATED)
A small scale model of a gas giant made of the same materials would fly apart in all directions. Jupiter does not explode because of gravity. Gravity pulls all matter towards all other matter (proportional to mass times inverse square of distance, you get the idea), and this tends to squash matter together until it is a sphere or near enough. Hydrogen molecules are still moving around randomly (with an average speed governed by temperature and gas laws) trying to escape but they don't get very far, gravity brings them back. Jupiter has the mass of 300 Earths, and has a firm gravitational grip on itself. A ball thrown away from Jupiter should return unless it is traveling at a critical speed called breaking point which is >60 km/s for Jupite